# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a Dataset object, see attributes below
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview

Review available record sets, fields, and their `@id`.

`mlcroissant` exposes record set information through the `record_sets` attribute on the dataset.
Each record set contains fields, with each field/column accessible via its own `@id` attribute.

Let's list all record sets and their fields.

In [ ]:
# List all record sets and fields by their @id
print("Available record sets and their fields (by @id):\n")
record_set_ids = []
for recordset in dataset.record_sets:
    print(f"- Record set: {recordset['@id']}")
    record_set_ids.append(recordset['@id'])
    fields = recordset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # ensure list
    for field in fields:
        # Each field is a dict referencing a column
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - Field/column: {field_id}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. All entities must be referenced by their `@id`.

Here's how to extract data from all available record sets.

In [ ]:
# Extract data from each available record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records.")
        print(f"Fields: {df.columns.tolist()}")
        print(df.head(2).to_string(index=False))
    except Exception as e:
        print(f"  Failed to load records: {e}")

# For the following analysis, select one main record set (the first):
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"\nExample record set selected for analysis: {selected_record_set_id}")
else:
    selected_record_set_id = None

# Display fields/columns for the selected record set
if selected_record_set_id:
    print(f"Available columns in selected record set: {dataframes[selected_record_set_id].columns.tolist()}")
    display_columns = dataframes[selected_record_set_id].columns.tolist()

## 4. Exploratory Data Analysis (EDA)

We will select a numeric field in the selected record set (by `@id`/column name) for filtering, normalization, and grouping operations.

* _Note_: You may wish to adapt the field ID to your dataset. We'll pick the first numeric-like column by inspecting the first row.

In [ ]:
# Inspect first record for numeric columns
import numpy as np
numeric_field = None
group_field = None
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Guess column types by sampling
    for col in df.columns:
        try:
            # Try to convert to numeric
            vals = pd.to_numeric(df[col], errors='coerce')
            if vals.notnull().sum() > 0:
                if numeric_field is None:
                    numeric_field = col
        except Exception:
            continue
    # Pick a groupable (string) field (not numeric, not Python object)
    for col in df.columns:
        if col != numeric_field:
            if not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                break

    print(f"Numeric field candidate (by @id/column name): {numeric_field}")
    print(f"Group field candidate (by @id/column name): {group_field}")

    # Ensure numeric conversion
    df_numeric = df.copy()
    df_numeric[numeric_field] = pd.to_numeric(df_numeric[numeric_field], errors='coerce')

    threshold = df_numeric[numeric_field].mean() if df_numeric[numeric_field].notnull().any() else 0
    filtered_df = df_numeric[df_numeric[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: (first 5 rows)")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"\nNormalized {numeric_field} for filtered records (first 5):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group and show means, if groupable field exists
    if group_field and group_field in df_numeric.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field} (top 5):")
        print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and compare grouped means (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field:
    plt.figure(figsize=(6, 4))
    sns.histplot(df_numeric[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df_numeric.columns:
        plt.figure(figsize=(8, 4))
        # Show top N categories only (for readability)
        top_groups = df_numeric[group_field].value_counts().head(10).index
        filtered_df_top = df_numeric[df_numeric[group_field].isin(top_groups)]
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df_top)
        plt.title(f"{numeric_field} by {group_field} (top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to explore and process the [FAIR² mass spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant`. We:
- Loaded the dataset schema and printed dataset-level metadata.
- Listed all available record sets and their fields using each entity's `@id`.
- Loaded record set data into DataFrames, filtering, normalizing, and grouping by column as referenced by `@id`.
- Visualized numeric distributions and category-specific summaries.

You may further customize field choices by inspecting the column names (`@id`) above.

_For detailed Croissant tutorials, see [mlcroissant documentation](https://mlcommons.github.io/croissant/python/)_